# Hankel-DMD Equation-Free Digital Twin: End-to-End Tutorial

This notebook walks through the full analysis workflow associated with the paper
**Equation-Free Digital Twins for Nonlinear Structural Dynamics** (arXiv:2605.00950).

The notebook is intentionally designed so that you can run it on the small
synthetic sample data shipped under `data/sample/` before placing the full
12-case OpenFAST export under `data/full/`. The synthetic data is **not**
suitable for reproducing the paper's R²>0.99 results — it only verifies that
the workflow loads, fits, and plots end-to-end without methodological
modification.

The cells below cover, in order:

1. Loading and inspecting OpenFAST-exported case data.
2. Hankel-DMD identification of fore-aft tower modes.
3. Eigenvalue stability map and physical mode-shape extraction.
4. Rolling-horizon virtual sensing reconstruction of hidden sensors.
5. SSI-COV cross-validation of identified modes.

If you have the full dataset placed at `data/full/Case_1.csv` ... `data/full/Case_12.csv`,
simply change the `DATA_DIR` constant below.


In [ ]:
# Standard imports and path bootstrap.
# We add `src/` to sys.path so the notebook works without installing the package.
import sys
from pathlib import Path

# Notebooks live one level below the repo root.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Make plots inline and a bit larger by default.
%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)

print('Repo root:', REPO_ROOT)


## 1. Locate and inspect the case files

The Hankel-DMD pipeline expects 12 OpenFAST-exported CSV files named
`Case_1.csv` ... `Case_12.csv`. For this tutorial we copy the synthetic
sample once into `data/full/` so the same loader code path runs in both
testing and full-paper mode.


In [ ]:
from eftwin.data_io import discover_case_files, read_case_csv
from eftwin.constants import analysis_columns

DATA_DIR = REPO_ROOT / 'data' / 'full'
SAMPLE_FILE = REPO_ROOT / 'data' / 'sample' / 'sample_case_small.csv'

# Bootstrap with the synthetic sample if data/full is empty.
DATA_DIR.mkdir(parents=True, exist_ok=True)
if not list(DATA_DIR.glob('Case_*.csv')):
    target = DATA_DIR / 'Case_1.csv'
    target.write_bytes(SAMPLE_FILE.read_bytes())
    print(f'Bootstrapped {target} from sample.')

case_files = discover_case_files(DATA_DIR)
print(f'Found {len(case_files)} case file(s):')
for f in case_files:
    print(' -', f.name)


In [ ]:
# Quick inspection of column names. The loader in `data_io.py` uses robust
# substring matching, so it tolerates the duplicate-column situation that
# OpenFAST sometimes produces for TwHt1MLxt and TwHt1MLyt.
df_first = read_case_csv(case_files[0])
print(f'Shape: {df_first.shape}')
print(f'First 5 columns: {list(df_first.columns[:5])}')
print(f'Last 5 columns: {list(df_first.columns[-5:])}')


## 2. Run Hankel-DMD identification on the fore-aft direction

The fore-aft analysis pairs nine tower acceleration channels (`TwHt*ALxt`)
with nine tower bending moments (`TwHt*MLyt`). Defaults below match the
original `Filter_Hankel_03.py` exactly: `dt=0.0125`, `hankel_d=60`, and
`svd_rank=24`.


In [ ]:
from eftwin.hankel_dmd import run_hankel_dmd

acc_cols, mom_cols = analysis_columns('fore_aft')
print('Acceleration columns:', acc_cols)
print('Moment columns:      ', mom_cols)

dmd_result = run_hankel_dmd(
    case_files,
    acc_cols,
    mom_cols,
    direction_name='Fore-Aft',
    dt=0.0125,
    hankel_d=60,
    svd_rank=24,
)
print(f'\nFitted Hankel-DMD model with rank {dmd_result.model.modes.shape[1]}.')
print(f'State matrix shape: {dmd_result.X_raw.shape}')


## 3. Eigenvalue stability map and physical mode shapes

The eigenvalue plot puts every Koopman eigenvalue on the complex unit circle,
where points inside the circle correspond to stable (decaying) modes and
points exactly on the circle to neutrally stable modes. Tower-candidate
modes near 0.5 Hz and near 2 Hz are highlighted.


In [ ]:
from eftwin.plotting import plot_eigs_custom, plot_mode_shapes
from eftwin.modal_analysis import extract_physical_modes

fig, ax = plot_eigs_custom(dmd_result, dt=0.0125)
plt.show()


In [ ]:
# Extract frequency and damping for modes inside the analysis band.
modes_df, stats_df = extract_physical_modes(dmd_result, dt=0.0125, low_f=0.3, high_f=3.0)
print('Identified modes (Hz, damping):')
print(stats_df)


In [ ]:
# Plot the corresponding tower-displacement mode shapes
fig, ax = plot_mode_shapes(dmd_result, dt=0.0125, low_f=0.3, high_f=3.0)
plt.show()


## 4. Virtual sensing: reconstruct hidden tower sensors

The virtual sensing routine simulates the failure of two tower-mounted
sensors (default: indices 0 and 8, which correspond to the lowest and
highest acceleration gauges) and recovers their time-series from the
remaining seven acceleration sensors plus the nine moment channels. This
matches the original `Virtual_sensing 2.py` algorithm.


In [ ]:
from eftwin.virtual_sensing import run_virtual_sensing
from eftwin.plotting import plot_virtual_sensing_result

vs_result = run_virtual_sensing(
    dmd_result,
    missing_sensors=(0, 8),
    update_every=1.0,
    duration=10.0,    # short window for the synthetic sample
    dt=0.0125,
)
print('Reconstruction metrics:')
print(vs_result['metrics'])

# Plot for the first hidden sensor.
fig, ax = plot_virtual_sensing_result(vs_result, sensor_index=0, plot_duration=10.0)
plt.show()


## 5. SSI-COV cross-validation

For the paper, the Hankel-DMD identification is cross-checked against a
classical covariance-driven Stochastic Subspace Identification (SSI-COV)
analysis. The numerical procedure here is exactly the one in `SSICOV_2.py`
— block-Hankel covariance, SVD, observability matrix, and eigendecomposition
of the resulting state-transition matrix.


In [ ]:
from eftwin.ssi_cov import run_ssi_validation

ssi_modes, ssi_stats, _ = run_ssi_validation(
    case_files,
    acc_cols,
    mom_cols,
    dt=0.0125,
    block_rows=60,
    model_order=40,
    downsample_factor=4,
)
print('SSI-COV identified modes:')
print(ssi_stats)


## What to do next

For the paper-equivalent results you will need:

* The full 12-case OpenFAST export placed at `data/full/Case_1.csv` ... `Case_12.csv`.
* The OpenFAST linearization mode-shape Excel file `Extracted_Mode_Shapes.xlsx`
  (produced by `legacy/openfast_original/Modal_OutPut_Final_02.py`).

Then, from the repository root, run the orchestration scripts:

```bash
python scripts/analysis/run_hankel_dmd.py --config configs/hankel_dmd.yaml
python scripts/analysis/run_virtual_sensing.py --config configs/virtual_sensing.yaml --direction fore_aft
python scripts/analysis/run_ssi_cov.py --config configs/ssi_cov.yaml --direction fore_aft
python scripts/analysis/run_probabilistic_modes.py --config configs/probabilistic_modes.yaml
python scripts/analysis/run_mac_validation.py --config configs/mac_validation.yaml
```

All cleaned scripts are documented in `docs/code_inventory.md` and
`docs/methodology.md`.
